In [ ]:
import gzip
import os
from scipy.stats import binomtest

# --- Paths ---
input_path  = "/n/data1/hms/dbmi/park/jiny/SMaHT/COLO829/TruthSet/Collection/0.calls_shortread/PU_longread/"
output_path = "/n/data1/hms/dbmi/park/jiny/SMaHT/COLO829/0.truthset/1.Illumina_PU20/"

# --- Output files for removed variants ---
out_removed     = open("MT.ins_remove.txt", 'w')      # insertions failing BL filter
out_del_removed = open("MT.del_remove.txt", 'w')      # deletions failing BL filter

dic_step1  = {}
dic_altzero = {}
dic_VAF    = {}   # keep VAFs for later annotation

# --- Modes to process ---
modes = ["MT", "STK", "VN", "RF"]

# --- Collect AF and OtherAlt from INFO field ---
def collect_AF(file):
    altcount_dict = {}
    otheralt_dict = {}
    line_dict     = {}

    for line in file:
        if not line or line[0] == '#':
            continue
        fields = line.rstrip("\n").split('\t')
        if len(fields) < 8:
            continue
        chrom, pos, ref, alt, filt, info_field = fields[0], fields[1], fields[3], fields[4], fields[6], fields[7]
        if filt == "PASS" or filt == '.':  # Use only PASS calls
            variant = f"{chrom}:{pos}:{ref}:{alt}"
            line_dict[variant] = line.strip()
            for element in info_field.split(';'):
                if element.startswith('DTV=') and 'POPAF' not in element:
                    try:
                        altcount_dict[variant] = float(element.split('=')[1])
                    except:
                        pass
                elif element.startswith('OtherAlt='):
                    otheralt_dict[variant] = element.split('=')[1].split('_')
    return altcount_dict, otheralt_dict, line_dict

# --- Build allele counts from OtherAlt ---
def compute_ranks_from_otheralt(ref, alt, otheralt_list):
    allele_counts = {}
    for entry in (otheralt_list or []):
        if '-' not in entry:
            continue
        a, c = entry.split('-', 1)
        try:
            cnt = int(c)
        except:
            continue
        allele_counts[a] = allele_counts.get(a, 0) + cnt
    allele_counts.setdefault(ref, 0)
    allele_counts.setdefault(alt, 0)

    def rank_of(a):
        ca = allele_counts.get(a, 0)
        return 1 + sum(1 for v in allele_counts.values() if v > ca)

    return allele_counts, rank_of(alt), rank_of(ref)

# --- Main loop ---
for mode in modes:
    tool = mode

    if mode in ["MT", "STK"]:
        chromosomes = [f"chr{i}" for i in range(1, 23)] + ["chrX", "chrY"]
    else:
        chromosomes = [None]

    total_count_all = alt_in_t_count_all = valid_count_all = alt_in_bl_count_all = alt_not_in_t_count_all = 0
    cnt_candi_ins = cnt_candi_del = cnt_valid_ins = cnt_valid_del = 0

    output_file_path = os.path.join(output_path, "Valid_tools_ind", f"{mode}.Valid.vcf")
    os.makedirs(os.path.dirname(output_file_path), exist_ok=True)

    with open(output_file_path, 'w') as output_file:
        for chrom in chromosomes:
            if chrom not in dic_step1:
                dic_step1[chrom] = {}
            if chrom not in dic_altzero:
                dic_altzero[chrom] = {}

            tumor_prefix = mode
            if chrom:
                tumor_suffix   = f"{chrom}_annoAllAlt_COLO829T_Hifi.vcf.gz"
                tumorSR_suffix = f"{chrom}_annoAllAlt_COLO829T_Ill_200X.vcf.gz"
                bloodSR_suffix = f"{chrom}_annoAllAlt_COLO829BL_Ill_230X.vcf.gz"
            else:
                tumor_suffix   = f"{mode}_annoAllAlt_COLO829T_Hifi.vcf.gz"
                tumorSR_suffix = f"{mode}_annoAllAlt_COLO829T_Ill_200X.vcf.gz"
                bloodSR_suffix = f"{mode}_annoAllAlt_COLO829BL_Ill_230X.vcf.gz"

            tumor_path   = os.path.join(input_path, tumor_prefix, tumor_suffix)
            tumorSR_path = os.path.join(input_path, tumor_prefix, tumorSR_suffix)
            bloodSR_path = os.path.join(input_path, tumor_prefix, bloodSR_suffix)

            with gzip.open(tumor_path, "rt") as tumor_file, \
                 gzip.open(tumorSR_path, "rt") as tumorSR_file, \
                 gzip.open(bloodSR_path, "rt") as bloodSR_file:

                tumor_alt, tumor_otheralt, tumor_lines     = collect_AF(tumor_file)
                tumorSR_alt, tumorSR_otheralt, _           = collect_AF(tumorSR_file)
                bloodSR_alt, bloodSR_otheralt, _           = collect_AF(bloodSR_file)

                total_count = alt_in_t_count = valid_count = alt_in_bl_count = alt_not_in_t_count = 0

                for variant in tumor_alt:
                    total_count += 1
                    parts = variant.split(':')
                    chrom_v, ref, alt = parts[0], parts[2], parts[3]

                    if len(ref) >= 51 or len(alt) >= 51:
                        continue

                    if len(ref) < len(alt):
                        cnt_candi_ins += 1
                    elif len(ref) > len(alt):
                        cnt_candi_del += 1

                    # --- PacBio tumor ---
                    allele_counts_pb, alt_rank, ref_rank = compute_ranks_from_otheralt(
                        ref, alt, tumor_otheralt.get(variant)
                    )
                    ref_count_pb = allele_counts_pb.get(ref, 0)
                    alt_count_pb = allele_counts_pb.get(alt, 0)
                    depth_pb     = sum(allele_counts_pb.values())
                    vaf_pb       = alt_count_pb / depth_pb if depth_pb > 0 else 0.0

                    # --- Illumina tumor ---
                    allele_counts_ill, _, _ = compute_ranks_from_otheralt(
                        ref, alt, tumorSR_otheralt.get(variant)
                    )
                    alt_count_ill = allele_counts_ill.get(alt, 0)
                    depth_ill     = sum(allele_counts_ill.values())
                    vaf_ill       = alt_count_ill / depth_ill if depth_ill > 0 else 0.0

                    # --- Illumina blood ---
                    allele_counts_bl, _, _ = compute_ranks_from_otheralt(
                        ref, alt, bloodSR_otheralt.get(variant)
                    )
                    alt_count_bl = allele_counts_bl.get(alt, 0)
                    depth_bl     = sum(allele_counts_bl.values())
                    vaf_bl       = alt_count_bl / depth_bl if depth_bl > 0 else 0.0

                    # --- Store VAFs ---
                    dic_VAF[variant] = {
                        "VAF_pb": vaf_pb,
                        "VAF_ill": vaf_ill,
                        "VAF_bl": vaf_bl
                    }

                    # --- Tumor balance check ---
                    tumor_balance_ok = False
                    if depth_pb > 0:
                        p_binom = binomtest(alt_count_pb, depth_pb, p=0.5, alternative='two-sided').pvalue
                        if vaf_pb > 0.5 or p_binom > 0.05:
                            tumor_balance_ok = True

                    # --- Filtering ---
                    if (vaf_pb > 0) and (alt_rank == 1 or (ref_rank == 1 and alt_rank == 2)) \
                       and vaf_pb > 0.2 and vaf_ill > 0.2:
                        alt_in_t_count += 1
                        if vaf_bl < 0.05:
                            valid_count += 1
                            dic_step1[chrom_v].setdefault(variant, []).append(tool)
                            output_file.write(f"{tumor_lines[variant]}\n")
                            if len(ref) < len(alt):
                                cnt_valid_ins += 1
                            elif len(ref) > len(alt):
                                cnt_valid_del += 1
                        else:
                            alt_in_bl_count += 1
                            if len(ref) < len(alt):
                                out_removed.write(variant + "\n")
                            elif len(ref) > len(alt):
                                out_del_removed.write(variant + "\n")
                    else:
                        alt_not_in_t_count += 1
                        dic_altzero[chrom_v].setdefault(variant, []).append(tool)

                total_count_all     += total_count
                alt_in_t_count_all  += alt_in_t_count
                valid_count_all     += valid_count
                alt_in_bl_count_all += alt_in_bl_count
                alt_not_in_t_count_all += alt_not_in_t_count

    # --- Print summary ---
    print(f"{mode} (Combined Chromosomes)")
    print(f"Total: {total_count_all}")
    print(f"Alt in T: {alt_in_t_count_all}")
    print(f"Alt not in T: {alt_not_in_t_count_all}")
    print(f"Valid: {valid_count_all}")
    print(f"Alt in BL (Illumina): {alt_in_bl_count_all}")
    print(f"Total, valid insertion: {cnt_candi_ins}, {cnt_valid_ins}")
    print(f"Total, valid deletion: {cnt_candi_del}, {cnt_valid_del}\n")

    os.makedirs(output_path, exist_ok=True)
    with open(os.path.join(output_path, "stats.txt"), 'a') as stats_file:
        stats_file.write(
            f"{tool}\tind\tCombined\t{total_count_all}\t{alt_in_t_count_all}\t"
            f"{valid_count_all}\t{alt_in_bl_count_all}\t{alt_not_in_t_count_all}\n"
        )
    print(f"{mode} done.")